In [7]:
import json
import duckdb
import pandas as pd
from pathlib import Path

# ==========================================
# SETUP DIRECTORY PATHS & DUCKDB CONNECTION
# ==========================================
BASE_DIR = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_DIR = BASE_DIR / "data_sets"
RESULT_DIR = BASE_DIR / "results"
ANALYSIS_DIR = BASE_DIR / "analysis"

DB_PATH = ANALYSIS_DIR / "step02_04_consistency_checks.duckdb"
con = duckdb.connect(str(DB_PATH))

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

FINDINGS = {}

delay_cleaned_path = RESULT_DIR / "combined_delay_cleaned.csv"
sched_imputed_path = RESULT_DIR / "combined_schedule_imputed.csv"
raw_delay_path = DATA_DIR / "combined_delay.csv"

print("Registering views for delay_cleaned, schedule_imputed, and raw_delay...")

con.execute(f"""
CREATE OR REPLACE VIEW delay_cleaned AS
SELECT 
    CAST(date AS DATE) AS date,
    CAST(train_no AS INTEGER) AS train_no,
    CAST(station_no AS INTEGER) AS station_no,
    CAST(station_name AS VARCHAR) AS station_name,
    CAST(delay AS DOUBLE) AS delay,
    CAST(is_imputed AS BOOLEAN) AS is_imputed
FROM read_csv_auto('{delay_cleaned_path.as_posix()}', header=True);

CREATE OR REPLACE VIEW schedule_imputed AS
SELECT 
    CAST(train_no AS INTEGER) AS train_no,
    CAST(station_no AS INTEGER) AS station_no,
    CAST(station_name AS VARCHAR) AS station_name,
    CAST(distance_from_origin AS DOUBLE) AS distance_from_origin,
    CAST(arrival_day AS INTEGER) AS arrival_day,
    CAST(arrival_time AS VARCHAR) AS arrival_time,
    CAST(departure_day AS INTEGER) AS departure_day,
    CAST(departure_time AS VARCHAR) AS departure_time,
    CAST(is_arrival_imputed AS BOOLEAN) AS is_arrival_imputed,
    CAST(is_departure_imputed AS BOOLEAN) AS is_departure_imputed,
    CAST(is_imputed AS BOOLEAN) AS is_schedule_imputed
FROM read_csv_auto('{sched_imputed_path.as_posix()}', header=True);

CREATE OR REPLACE VIEW raw_delay AS
SELECT 
    CAST(date AS DATE) AS date,
    CAST(train_no AS INTEGER) AS train_no,
    CAST(station_no AS INTEGER) AS station_no,
    CAST(station_name AS VARCHAR) AS station_name,
    TRY_CAST(delay AS DOUBLE) AS delay
FROM read_csv_auto('{raw_delay_path.as_posix()}', header=True);
""")

counts_df = q("""
SELECT 
    (SELECT COUNT(*) FROM delay_cleaned) AS delay_cleaned_rows,
    (SELECT COUNT(*) FROM schedule_imputed) AS schedule_imputed_rows,
    (SELECT COUNT(*) FROM raw_delay) AS raw_delay_rows;
""")
print(counts_df.to_string(index=False))
FINDINGS["row_counts"] = counts_df.iloc[0].to_dict()

Registering views for delay_cleaned, schedule_imputed, and raw_delay...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 delay_cleaned_rows  schedule_imputed_rows  raw_delay_rows
           36614824                 172112        38428703


In [8]:
# ==========================================
# CHECK 1: KEY UNIQUENESS
# ==========================================
print("\n=== CHECK 1: KEY UNIQUENESS ===")

sched_dup_df = q("""
SELECT COUNT(*) AS dup_schedule_keys
FROM (
    SELECT train_no, station_no
    FROM schedule_imputed
    GROUP BY train_no, station_no
    HAVING COUNT(*) > 1
);
""")

delay_dup_df = q("""
SELECT COUNT(*) AS dup_delay_keys
FROM (
    SELECT date, train_no, station_no
    FROM delay_cleaned
    GROUP BY date, train_no, station_no
    HAVING COUNT(*) > 1
);
""")

uniqueness_res = {
    "duplicated_schedule_keys_train_station": int(sched_dup_df["dup_schedule_keys"][0]),
    "duplicated_delay_keys_date_train_station": int(delay_dup_df["dup_delay_keys"][0])
}
print(pd.DataFrame([uniqueness_res]).to_string(index=False))
FINDINGS["check1_key_uniqueness"] = uniqueness_res


=== CHECK 1: KEY UNIQUENESS ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 duplicated_schedule_keys_train_station  duplicated_delay_keys_date_train_station
                                      0                                         0


In [9]:
# ==========================================
# CHECK 2: JOIN INTEGRITY ON (train_no, station_no)
# ==========================================
print("\n=== CHECK 2: JOIN INTEGRITY (LEFT JOIN ON train_no, station_no) ===")

join_integrity_df = q("""
SELECT 
    COUNT(*) AS total_joined_rows,
    COUNT(*) FILTER (WHERE s.station_no IS NULL) AS unmatched_schedule_stop_rows,
    ROUND(COUNT(*) FILTER (WHERE s.station_no IS NULL) * 100.0 / COUNT(*), 4) AS unmatched_schedule_stop_pct,
    COUNT(*) FILTER (WHERE s.station_no IS NOT NULL AND d.station_name != s.station_name) AS station_name_mismatch_rows,
    ROUND(COUNT(*) FILTER (WHERE s.station_no IS NOT NULL AND d.station_name != s.station_name) * 100.0 / COUNT(*), 4) AS station_name_mismatch_pct,
    COUNT(*) FILTER (WHERE s.station_no IS NOT NULL AND d.station_name = s.station_name) AS exact_match_rows,
    ROUND(COUNT(*) FILTER (WHERE s.station_no IS NOT NULL AND d.station_name = s.station_name) * 100.0 / COUNT(*), 4) AS exact_match_pct
FROM delay_cleaned d
LEFT JOIN schedule_imputed s
  ON d.train_no = s.train_no
 AND d.station_no = s.station_no;
""")

print(join_integrity_df.to_string(index=False))
FINDINGS["check2_join_integrity"] = join_integrity_df.iloc[0].to_dict()


=== CHECK 2: JOIN INTEGRITY (LEFT JOIN ON train_no, station_no) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_joined_rows  unmatched_schedule_stop_rows  unmatched_schedule_stop_pct  station_name_mismatch_rows  station_name_mismatch_pct  exact_match_rows  exact_match_pct
          36614824                             0                          0.0                     6596040                    18.0147          30018784          81.9853


In [10]:
# ==========================================
# CHECK 3: DAY CONSISTENCY (arrival_day vs delay date)
# ==========================================
print("\n=== CHECK 3A: SCHEDULE MULTI-DAY STOP SHARE & JOURNEY COMPLETENESS ===")

sched_multiday_df = q("""
SELECT 
    COUNT(*) AS total_sched_stops,
    COUNT(*) FILTER (WHERE arrival_day > 1) AS stops_after_day1,
    ROUND(COUNT(*) FILTER (WHERE arrival_day > 1) * 100.0 / COUNT(*), 4) AS pct_stops_after_day1
FROM schedule_imputed;
""")
print("Schedule Multi-Day Stop Share:")
print(sched_multiday_df.to_string(index=False))

# Journey completeness per (date, train_no) classified by whether the train is single-day or multi-day in schedule
journey_completeness_df = q("""
WITH TrainMaxDay AS (
    SELECT 
        train_no, 
        MAX(arrival_day) AS max_arrival_day,
        CASE WHEN MAX(arrival_day) > 1 THEN 'multi_day' ELSE 'single_day' END AS journey_type
    FROM schedule_imputed
    GROUP BY train_no
),
JourneyGroups AS (
    SELECT 
        d.date,
        d.train_no,
        COALESCE(t.journey_type, 'unscheduled') AS journey_type,
        MIN(d.station_no) AS min_stop,
        MAX(d.station_no) AS max_stop,
        COUNT(*) AS n_stops
    FROM delay_cleaned d
    LEFT JOIN TrainMaxDay t ON d.train_no = t.train_no
    GROUP BY d.date, d.train_no, t.journey_type
)
SELECT 
    journey_type,
    COUNT(*) AS total_journeys,
    COUNT(*) FILTER (WHERE min_stop = 1) AS starts_at_stop1,
    ROUND(COUNT(*) FILTER (WHERE min_stop = 1) * 100.0 / COUNT(*), 4) AS pct_starts_at_stop1,
    COUNT(*) FILTER (WHERE (max_stop - min_stop + 1) = n_stops) AS contiguous_journeys,
    ROUND(COUNT(*) FILTER (WHERE (max_stop - min_stop + 1) = n_stops) * 100.0 / COUNT(*), 4) AS pct_contiguous
FROM JourneyGroups
GROUP BY journey_type
ORDER BY journey_type;
""")
print("\nJourney Completeness per (date, train_no):")
print(journey_completeness_df.to_string(index=False))

print("\n=== CHECK 3B: PERSISTENCE CORRELATION TEST (ORIGIN DELAY ON D vs FINAL STOP DELAY ON D+k) ===")
# Demeaned correlation between origin delay on date D and final stop delay on date D+k (k = 0, 1, 2, 3)
persistence_df = q("""
WITH TrainSpan AS (
    SELECT 
        train_no,
        MAX(station_no) AS final_sched_stop,
        MAX(arrival_day) AS max_arrival_day,
        MAX(arrival_day) - 1 AS transit_days
    FROM schedule_imputed
    GROUP BY train_no
    HAVING MAX(arrival_day) BETWEEN 1 AND 4
),
JourneyEndpoints AS (
    SELECT 
        d.date,
        d.train_no,
        ts.max_arrival_day,
        ts.transit_days,
        MAX(CASE WHEN d.station_no = 1 THEN d.delay END) AS origin_delay,
        MAX(CASE WHEN d.station_no = ts.final_sched_stop THEN d.delay END) AS final_delay
    FROM delay_cleaned d
    JOIN TrainSpan ts ON d.train_no = ts.train_no
    WHERE d.is_imputed = FALSE
      AND (d.station_no = 1 OR d.station_no = ts.final_sched_stop)
    GROUP BY d.date, d.train_no, ts.max_arrival_day, ts.transit_days
),
TrainMeans AS (
    SELECT 
        train_no,
        AVG(origin_delay) AS mean_origin_delay,
        AVG(final_delay) AS mean_final_delay
    FROM JourneyEndpoints
    GROUP BY train_no
),
Demeaned AS (
    SELECT 
        j.date,
        j.train_no,
        j.max_arrival_day,
        j.transit_days,
        j.origin_delay - m.mean_origin_delay AS orig_dm,
        j.final_delay - m.mean_final_delay AS final_dm
    FROM JourneyEndpoints j
    JOIN TrainMeans m ON j.train_no = m.train_no
    WHERE j.origin_delay IS NOT NULL OR j.final_delay IS NOT NULL
)
SELECT 
    a.max_arrival_day AS schedule_days,
    a.transit_days AS expected_offset_if_calendar_date,
    ROUND(CORR(a.orig_dm, b0.final_dm), 4) AS corr_k0_same_date,
    ROUND(CORR(a.orig_dm, b1.final_dm), 4) AS corr_k1_plus_1d,
    ROUND(CORR(a.orig_dm, b2.final_dm), 4) AS corr_k2_plus_2d,
    ROUND(CORR(a.orig_dm, b3.final_dm), 4) AS corr_k3_plus_3d,
    COUNT(b0.final_dm) AS n_pairs_k0
FROM Demeaned a
LEFT JOIN Demeaned b0 ON a.train_no = b0.train_no AND b0.date = a.date
LEFT JOIN Demeaned b1 ON a.train_no = b1.train_no AND b1.date = a.date + INTERVAL 1 DAY
LEFT JOIN Demeaned b2 ON a.train_no = b2.train_no AND b2.date = a.date + INTERVAL 2 DAY
LEFT JOIN Demeaned b3 ON a.train_no = b3.train_no AND b3.date = a.date + INTERVAL 3 DAY
WHERE a.orig_dm IS NOT NULL
GROUP BY a.max_arrival_day, a.transit_days
ORDER BY a.max_arrival_day;
""")
print(persistence_df.to_string(index=False))

FINDINGS["check3_day_consistency"] = {
    "schedule_multiday_share": sched_multiday_df.iloc[0].to_dict(),
    "journey_completeness": journey_completeness_df.to_dict(orient="records"),
    "persistence_correlations": persistence_df.to_dict(orient="records")
}


=== CHECK 3A: SCHEDULE MULTI-DAY STOP SHARE & JOURNEY COMPLETENESS ===
Schedule Multi-Day Stop Share:
 total_sched_stops  stops_after_day1  pct_stops_after_day1
            172112             48134               27.9667


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Journey Completeness per (date, train_no):
journey_type  total_journeys  starts_at_stop1  pct_starts_at_stop1  contiguous_journeys  pct_contiguous
   multi_day          486982           486308              99.8616               479960         98.5581
  single_day         1365062          1365041              99.9985              1364868         99.9858

=== CHECK 3B: PERSISTENCE CORRELATION TEST (ORIGIN DELAY ON D vs FINAL STOP DELAY ON D+k) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 schedule_days  expected_offset_if_calendar_date  corr_k0_same_date  corr_k1_plus_1d  corr_k2_plus_2d  corr_k3_plus_3d  n_pairs_k0
             1                                 0             0.4715           0.1281           0.0928           0.0967     1288954
             2                                 1             0.6131           0.2489           0.2157           0.1920      373456
             3                                 2             0.5262           0.2024           0.1553           0.1348       60556
             4                                 3             0.4363          -0.0006           0.0118           0.0612        3496


In [11]:
# ==========================================
# CHECK 4: IMPUTED ROW FLAG RECONCILIATION & EXPORT
# ==========================================
print("\n=== CHECK 4: IMPUTED ROW FLAG RECONCILIATION ===")

# Cross-check `is_imputed` in delay_cleaned against NULL delays in raw_delay
reconcile_df = q("""
WITH RawStats AS (
    SELECT 
        COUNT(*) AS raw_total_rows,
        COUNT(*) FILTER (WHERE delay IS NULL) AS raw_null_rows
    FROM raw_delay
),
CleanedStats AS (
    SELECT 
        COUNT(*) AS cleaned_total_rows,
        COUNT(*) FILTER (WHERE is_imputed = TRUE) AS cleaned_imputed_rows,
        COUNT(*) FILTER (WHERE is_imputed = FALSE) AS cleaned_observed_rows,
        ROUND(COUNT(*) FILTER (WHERE is_imputed = TRUE) * 100.0 / COUNT(*), 4) AS pct_imputed_in_cleaned
    FROM delay_cleaned
),
FlagCrossCheck AS (
    SELECT 
        COUNT(*) FILTER (WHERE d.is_imputed = TRUE AND r.delay IS NOT NULL) AS imputed_flag_but_raw_not_null,
        COUNT(*) FILTER (WHERE d.is_imputed = FALSE AND r.delay IS NULL) AS observed_flag_but_raw_is_null
    FROM delay_cleaned d
    JOIN raw_delay r
      ON d.date = r.date AND d.train_no = r.train_no AND d.station_no = r.station_no
)
SELECT 
    r.raw_total_rows,
    r.raw_null_rows,
    c.cleaned_total_rows,
    c.cleaned_imputed_rows,
    c.cleaned_observed_rows,
    c.pct_imputed_in_cleaned,
    (r.raw_null_rows - c.cleaned_imputed_rows) AS raw_nulls_absent_from_cleaned,
    f.imputed_flag_but_raw_not_null,
    f.observed_flag_but_raw_is_null
FROM RawStats r, CleanedStats c, FlagCrossCheck f;
""")
print(reconcile_df.to_string(index=False))

# Export imputed_delay_keys.csv so test targets can also filter via key lookup if needed
imputed_keys_file = RESULT_DIR / "imputed_delay_keys.csv"
con.execute(f"""
COPY (
    SELECT date, train_no, station_no
    FROM delay_cleaned
    WHERE is_imputed = TRUE
    ORDER BY date, train_no, station_no
) TO '{imputed_keys_file.as_posix()}' (HEADER, DELIMITER ',');
""")
print(f"\n-> Exported {reconcile_df['cleaned_imputed_rows'][0]:,} imputed keys to {imputed_keys_file.as_posix()}")

FINDINGS["check4_imputed_row_reconciliation"] = reconcile_df.iloc[0].to_dict()

# Save Step 2.4 findings JSON
findings_path = RESULT_DIR / "step02_04_consistency_FINDINGS.json"
with open(findings_path, "w") as f:
    json.dump(FINDINGS, f, indent=4)
print(f"-> Saved findings to {findings_path.as_posix()}")

con.close()


=== CHECK 4: IMPUTED ROW FLAG RECONCILIATION ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 raw_total_rows  raw_null_rows  cleaned_total_rows  cleaned_imputed_rows  cleaned_observed_rows  pct_imputed_in_cleaned  raw_nulls_absent_from_cleaned  imputed_flag_but_raw_not_null  observed_flag_but_raw_is_null
       38428703        1874271            36614824               1032506               35582318                  2.8199                         841765                              0                              0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


-> Exported 1,032,506 imputed keys to D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/imputed_delay_keys.csv
-> Saved findings to D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/step02_04_consistency_FINDINGS.json
